# 第 16 章：量化与部署

这个 notebook 对应 `lessons/16_quantization_and_serving.md`，用离线 fake generator 演示 API 契约、generation config、benchmark、量化版本对比和 rollback 配置。

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from src.data.text_datasets import ChatMessage
from src.serving.runtime import (
    BenchmarkResult,
    DeploymentConfig,
    GeneratedOutput,
    GenerationConfig,
    LocalServingEngine,
    QuantizationRun,
    ServingRequest,
    benchmark_requests,
    validate_deployment_config,
    write_quantization_report,
)

## 1. 部署配置与 API 契约

部署配置必须保留 model version、RAG index version、量化方式和 rollback target。

In [ ]:
deployment = DeploymentConfig(
    model_version="legal-sft-v2-int8",
    rollback_target="legal-sft-v1-fp16",
    rag_index_version="legal-guidelines-2026-05",
    quantization="int8",
    owner="serving-owner",
    max_concurrency=4,
)
validate_deployment_config(deployment)
deployment.to_dict()

## 2. 本地 Serving Engine

教学版 engine 不启动网络服务，而是用函数模拟模型推理，重点检查 response 字段。

In [ ]:
def fake_generate(messages, config):
    return GeneratedOutput(
        answer=f"风险提示：需要人工复核。max_new_tokens={config.max_new_tokens}",
        citations=["contract#chunk_1"],
        safety_flags=["needs_human_review"],
        output_tokens=8,
    )

engine = LocalServingEngine(deployment, fake_generate)
request = ServingRequest(
    request_id="req_001",
    messages=[ChatMessage(role="user", content="分析这段合同风险")],
    generation_config=GenerationConfig(max_new_tokens=64, temperature=0.2),
)

response = engine.handle(request)
response.to_dict()

## 3. Benchmark

Benchmark 至少报告 p50 / p95 latency、tokens/s 和 error rate。

In [ ]:
requests = [
    ServingRequest(
        request_id=f"req_{i}",
        messages=[ChatMessage(role="user", content="分析合同风险")],
        generation_config=GenerationConfig(max_new_tokens=32),
    )
    for i in range(3)
]

benchmark = benchmark_requests(engine, requests)
benchmark.to_dict()

## 4. 量化版本对比

fp16、int8、int4 必须跑同一 eval set，报告质量、延迟、吞吐和显存。

In [ ]:
runs = [
    QuantizationRun(
        version="legal-fp16",
        precision="fp16",
        memory_mb=8000,
        benchmark=BenchmarkResult(3, 0, 24, 35, 40),
        eval_metrics={"json_valid": 1.0, "safe_refusal": 0.95},
        eval_set_id="legal_eval_v1",
    ),
    QuantizationRun(
        version="legal-int8",
        precision="int8",
        memory_mb=4200,
        benchmark=BenchmarkResult(3, 0, 18, 28, 55),
        eval_metrics={"json_valid": 1.0, "safe_refusal": 0.95},
        eval_set_id="legal_eval_v1",
    ),
    QuantizationRun(
        version="legal-int4",
        precision="int4",
        memory_mb=2600,
        benchmark=BenchmarkResult(3, 0, 16, 26, 60),
        eval_metrics={"json_valid": 0.96, "safe_refusal": 0.90},
        eval_set_id="legal_eval_v1",
    ),
]

with TemporaryDirectory() as tmpdir:
    report_path = Path(tmpdir) / "quantization_report.md"
    write_quantization_report(report_path, runs)
    print(report_path.read_text())